# 05 — CRS Limitations

Every buffer we have built so far uses the **destination point formula** — kilometer-accurate, spherically correct. But many real-world analyses skip this and use **degree offsets** instead:

```python
# naive — wrong
lon + r * cos(angle),  lat + r * sin(angle)
```

This looks like a circle on the screen, but it is an **oval** in the real world. The distortion is invisible at low latitudes and severe at high latitudes — which means it fails exactly in the places (Arctic, Northern Europe, Russia, Canada) where precision matters most.

## Setup

In [ ]:
import math
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps


# Kilometer-accurate buffer (spherical Earth)
def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def km_buffer(lon, lat, radius_km, n_points=64):
    """True circular buffer in kilometers."""
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

def degree_buffer(lon, lat, radius_deg, n_points=64):
    """Naive degree-offset buffer — oval in reality."""
    ring = []
    for i in range(n_points):
        angle = 2 * math.pi * i / n_points
        ring.append([lon + radius_deg * math.cos(angle),
                     lat + radius_deg * math.sin(angle)])
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

print("Both buffer functions defined.")

## Why Degrees Are Not Meters

The Earth is a sphere. One degree of **latitude** is always ≈ 111 km — latitude lines are equally spaced.

One degree of **longitude** depends on where you are:

```
1° longitude ≈ 111 × cos(latitude) km
```

| Latitude | 1° longitude ≈ |
|---|---|
| 0° (equator) | 111 km |
| 20° | 104 km |
| 35° | 91 km |
| 45° | 78 km |
| 55° | 64 km |
| 60° | 56 km |
| 70° | 38 km |

A degree-offset circle uses the same number of degrees in every direction. But each degree of longitude represents fewer kilometers as latitude increases — so the circle is stretched east-west compared to its true km size.

In [ ]:
# Show the shrinking km/degree relationship
print(f"{'Latitude':>10}  {'1° lon (km)':>12}  {'E-W stretch factor':>20}")
print("─" * 48)
for lat in [0, 20, 35, 45, 55, 60, 70]:
    km_per_deg = 111.0 * math.cos(math.radians(lat))
    # stretch: how many km does 1° lon represent vs 1° lat (111 km)
    stretch = 111.0 / km_per_deg if km_per_deg > 0 else float('inf')
    print(f"  {lat:>7}°  {km_per_deg:>11.1f}  {stretch:>19.2f}x")

## Side-by-Side Comparison: Equator vs High Latitude

To make the distortion visible, we overlay the naive degree buffer (red) and the km-accurate buffer (blue) at the same location.

- At the equator: nearly identical
- At 60°N: the degree buffer is visibly wider east-west than the km buffer

We choose a `radius_deg` such that the degree buffer has roughly the same *north-south* extent as the km buffer (since 1° lat ≈ 111 km, `radius_deg ≈ radius_km / 111`).

In [ ]:
RADIUS_KM  = 500
RADIUS_DEG = RADIUS_KM / 111.0   # match north-south extent

# --- Equator comparison ---
eq_lon, eq_lat = 20.0, 0.0

eq_km_feat  = {"type": "Feature", "properties": {"label": "km buffer"},
               "geometry": km_buffer(eq_lon, eq_lat, RADIUS_KM)}
eq_deg_feat = {"type": "Feature", "properties": {"label": "degree buffer"},
               "geometry": degree_buffer(eq_lon, eq_lat, RADIUS_DEG)}

m_eq = Map(center=(eq_lat, eq_lon), zoom=4, basemap=basemaps.CartoDB.Positron)
m_eq.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [eq_deg_feat]},
    name="Degree buffer (equator)",
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.2, "weight": 2}
))
m_eq.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [eq_km_feat]},
    name="km buffer (equator)",
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.2, "weight": 2}
))
m_eq.add(LayersControl())

print(f"Equator (0°): radius_deg={RADIUS_DEG:.2f}°  radius_km={RADIUS_KM} km")
print("At the equator, the two buffers are nearly identical.")
m_eq

In [ ]:
# --- 60°N comparison ---
hi_lon, hi_lat = 20.0, 60.0

hi_km_feat  = {"type": "Feature", "properties": {"label": "km buffer"},
               "geometry": km_buffer(hi_lon, hi_lat, RADIUS_KM)}
hi_deg_feat = {"type": "Feature", "properties": {"label": "degree buffer"},
               "geometry": degree_buffer(hi_lon, hi_lat, RADIUS_DEG)}

m_hi = Map(center=(hi_lat, hi_lon), zoom=4, basemap=basemaps.CartoDB.Positron)
m_hi.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [hi_deg_feat]},
    name="Degree buffer (60°N)",
    style={"color": "#e74c3c", "fillColor": "#e74c3c", "fillOpacity": 0.2, "weight": 2}
))
m_hi.add(GeoJSON(
    data={"type": "FeatureCollection", "features": [hi_km_feat]},
    name="km buffer (60°N)",
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.2, "weight": 2}
))
m_hi.add(LayersControl())

# Compute actual E-W extent of each buffer
km_per_lon_at_60 = 111.0 * math.cos(math.radians(60))
km_ew_deg  = RADIUS_DEG * 2 * 111.0       # degree buffer E-W in km equivalent
km_ew_km   = RADIUS_KM * 2                # true km buffer E-W
print(f"At 60°N: 1° lon ≈ {km_per_lon_at_60:.1f} km")
print(f"Degree buffer E-W span ≈ {km_ew_deg:.0f} km (as measured N-S)")
print(f"km buffer E-W span = {km_ew_km} km (correct)")
print(f"Degree buffer is ~{km_ew_deg/km_ew_km:.1f}x wider than the real buffer E-W")
m_hi

## Quantifying the Error

For a 500 km buffer, the E-W error of the naive approach grows rapidly with latitude. We measure this as the *east-west excess* — how many km wider the degree buffer is compared to the true km buffer.

In [ ]:
R_KM  = 500   # km
R_DEG = R_KM / 111.0

print(f"Buffer radius: {R_KM} km  ({R_DEG:.3f}°)")
print()
print(f"{'Latitude':>10}  {'km/1° lon':>10}  {'Deg-buf E-W (km)':>18}  {'True E-W (km)':>14}  {'Error':>8}")
print("─" * 68)
for lat in [0, 10, 20, 30, 45, 55, 60, 70]:
    km_per_lon  = 111.0 * math.cos(math.radians(lat))
    deg_ew_km   = R_DEG * 2 * km_per_lon   # actual km spanned by degree buffer in E-W direction
    true_ew_km  = R_KM * 2
    error_pct   = 100.0 * (true_ew_km - deg_ew_km) / true_ew_km
    print(f"  {lat:>7}°  {km_per_lon:>9.1f}  {deg_ew_km:>17.0f}  {true_ew_km:>13.0f}  {error_pct:>6.1f}%")
print()
print("Negative error means degree buffer is NARROWER than true in E-W (km/deg < 111).")
print("The degree buffer E-W is shrinking while N-S stays constant → oval, not circle.")

## When Does It Matter?

The distortion is proportional to latitude:

| Use case | Typical latitude | Degree buffer error |
|---|---|---|
| Middle East targets (Tehran, Riyadh) | 25–36°N | ~10–20% |
| European targets (Madrid, Berlin) | 40–52°N | ~25–35% |
| High-latitude (Moscow, Oslo) | 55–60°N | ~40–50% |
| Arctic scenarios | 70–80°N | >60% |

A 500 km lethal zone modeled with a degree offset could be 200 km too narrow east-west at 60°N — which would exclude cities that are genuinely within range. For analysis that affects real decisions, this is an unacceptable error.

## Exercise A

Compare degree-offset vs km-accurate 300 km buffers at **three latitudes**: 10°N, 35°N, and 65°N (all at the same longitude, e.g. 30°E).

1. Create a map showing all six buffers (two per latitude, different colors per method).
2. Print the E-W span in km for the degree buffer and the km buffer at each latitude.
3. At which latitude does the distortion first exceed 25%?

In [ ]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

RADIUS_KM = 300
LON = 30.0
LATITUDES = [10, 35, 65]

# For each latitude:
#   - compute radius_deg = RADIUS_KM / 111
#   - create degree_buffer (red) and km_buffer (blue)
#   - display all 6 on one map
# Print E-W span comparison and identify >25% distortion threshold
# Your code here

## Exercise B

**Moscow** is at approximately `[37.617, 55.755]`.

1. Create a 600 km degree-offset buffer around Moscow.
2. Create a 600 km km-accurate buffer around Moscow.
3. Display both on the same map and describe the shape difference.
4. Calculate: if you used the degree buffer to decide whether a city is within 600 km, could you incorrectly *exclude* a city that is actually within range? Which direction would the error occur?

In [ ]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

moscow = [37.617, 55.755]

# 1. 600 km degree buffer around Moscow
# 2. 600 km km buffer around Moscow
# 3. Display both — observe shape
# 4. Explain directional error (east-west vs north-south)
# Your code here

---

## Check Your Understanding

**1.** Why does the degree-offset buffer produce an oval rather than a circle, and in which direction is it distorted?

**2.** At the equator, a degree buffer is nearly correct. At 60°N it is badly wrong. What changes between those two latitudes that causes this?

```python
# No code needed — answer in your own words
```

## Next

In [06 — Applications: Impact Zones](./06-Applications_Impact_Zones.ipynb), we apply everything from this module to a complete scenario: blast radii around targets, trajectory corridors, and identifying which locations fall within a given impact zone.